In [1]:
!pip install transformers sentencepiece scikit-learn -q

In [18]:
import os
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BertTokenizer, BertForSequenceClassification
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
import time
from tqdm import tqdm
import torch.nn.functional as F
import re

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(device)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [4]:
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度'
os.chdir(drive_path)

Mounted at /content/drive


In [40]:
def add_exchange_suffix(code: str) -> str:
    code = str(code)

    if len(code) == 5 and code.isdigit():
        ticker = f"{code}.HK"
        return ticker

    if code.startswith(("60", "68", "900")):
        ticker = f"{code}.SH"
        return ticker

    if code.startswith(("00", "30", "200")):
        ticker = f"{code}.SZ"
        return ticker

    return code


def text_pre_process(text: str) -> str:
    if text is None or not isinstance(text, str):
        return text

    # --- Currency ---
    text = text.replace("人民币", "RMB")
    text = text.replace("亿元", " billion RMB")
    text = text.replace("百万元", " million RMB")
    text = text.replace("万元", " ten-thousand RMB")
    text = text.replace("元", " RMB")

    # --- Percentages ---
    text = text.replace("（", "(").replace("）", ")")  # normalize brackets
    text = re.sub(r"(\d+)%", r"\1 %", text)           # space before %

    text = re.sub(r"¥+", " RMB", text)                # repeated ¥

    return text.strip()


def process_report(ID, title_cn, summary_cn, pre_processing = False):
    if pre_processing:
        title_cn   = text_pre_process(title_cn)
        summary_cn = text_pre_process(summary_cn)

    title_en = translator(title_cn, max_length=600, truncation=True)[0]['translation_text'] if pd.notna(title_cn) else None
    summary_en = translator(summary_cn, max_length=1024, truncation=True)[0]['translation_text'] if pd.notna(summary_cn) else None

    if title_en is not None:
        inputs = tokenizer(title_en, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            title_logits = model(**inputs).logits.squeeze().cpu().numpy()
            title_probs = F.softmax(torch.tensor(title_logits), dim=-1).numpy()
    else:
        title_logits = [None, None, None]
        title_probs = [None, None, None]

    if summary_en is not None:
        inputs = tokenizer(summary_en, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            summary_logits = model(**inputs).logits.squeeze().cpu().numpy()
            summary_probs = F.softmax(torch.tensor(summary_logits), dim=-1).numpy()
    else:
        summary_logits = [None, None, None]
        summary_probs = [None, None, None]

    translation_dict = {
        "REPORT_ID": ID,
        "TITLE_EN": title_en,
        "SUMMARY_EN": summary_en
    }

    logit_dict = {
        "REPORT_ID": ID,
        "title_neg_logit": title_logits[0],
        "title_neu_logit": title_logits[1],
        "title_pos_logit": title_logits[2],
        "summary_neg_logit": summary_logits[0],
        "summary_neu_logit": summary_logits[1],
        "summary_pos_logit": summary_logits[2],
    }

    softmax_dict = {
        "REPORT_ID": ID,
        "title_neg_prob": title_probs[0],
        "title_neu_prob": title_probs[1],
        "title_pos_prob": title_probs[2],
        "summary_neg_prob": summary_probs[0],
        "summary_neu_prob": summary_probs[1],
        "summary_pos_prob": summary_probs[2],
    }

    return translation_dict, logit_dict, softmax_dict

In [42]:
equity_reports = pd.read_csv(
    'equity_reports_1.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE'],
    dtype={"STOCK_CODE": str}
)

equity_reports['PUBLISH_DATE'] = pd.to_datetime(equity_reports['PUBLISH_DATE'], errors='coerce')
equity_reports['UPDATE_TIME']  = pd.to_datetime(equity_reports['UPDATE_TIME'], errors='coerce')
equity_reports['UPDATE_DATE']  = pd.to_datetime(equity_reports['UPDATE_DATE'], errors='coerce')

equity_reports['STOCK_CODE'] = equity_reports['STOCK_CODE'].apply(add_exchange_suffix)

print(len(equity_reports))

10874


In [7]:
nllb_model_name = "facebook/nllb-200-3.3B"
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

translator = pipeline(
    "translation",
    model=nllb_model,
    tokenizer = nllb_tokenizer,
    src_lang = "zho_Hans",
    tgt_lang = "eng_Latn",
    device=device,
    truncation = True,
    max_length = 512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model-00003-of-00003.bin:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/8.55G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
finbert_model = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(finbert_model)
model = BertForSequenceClassification.from_pretrained(finbert_model).to(device)
model.eval()  # inference mode

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [43]:
row = equity_reports.iloc[0]
ID = row['REPORT_ID']
title_cn = row['TITLE']
summary_cn = row['SUMMARY']

translation_dict, logit_dict, softmax_dict = process_report(ID, title_cn, summary_cn, pre_processing=False)

print("=== Original Chinese ===")
print("TITLE_CN :", title_cn)
print("SUMMARY_CN:", summary_cn)

print("\n=== Translated English ===")
print("TITLE_EN :", translation_dict["TITLE_EN"])
print("SUMMARY_EN:", translation_dict["SUMMARY_EN"])

print("\n=== Logit Dict ===")
for k, v in logit_dict.items():
    print(f"{k}: {v}")

print("\n=== Softmax Dict ===")
for k, v in softmax_dict.items():
    print(f"{k}: {v}")

assert translation_dict["REPORT_ID"] == ID
assert isinstance(translation_dict["TITLE_EN"], str) or translation_dict["TITLE_EN"] is None
assert isinstance(translation_dict["SUMMARY_EN"], str) or translation_dict["SUMMARY_EN"] is None
assert len([k for k in logit_dict if "logit" in k]) == 6
assert len([k for k in softmax_dict if "prob" in k]) == 6

print("\n Unit test passed for row 0")

=== Original Chinese ===
TITLE_CN : 巴比食品（605338）：业绩目标稳健，完善激励机制
SUMMARY_CN: 巴比食品(605338)事件：公司发布《2022年限制性股票激励计划》，拟向激励对象授予的限制性股票数量为240.925万股，占本激励计划草案公告时公司股本总额24,800万股的0.97%。其中，首次授予220.925万股，占本激励计划公告时公司股本总额24,800万股的0.89%；预留20.00万股，占本激励计划公告时公司股本总额24,800万股的0.08%。点评：激励股份份额高，业绩目标设置稳健。公司此次限制性股份的激励份额占比近1%，激励对象涉及董事李俊、董秘/财务总监苏爽及中层管理人员及核心技术（业务）骨干等123人，覆盖激励对象范围广。授予价格15.51元/股，为公告前交易日均价的50%，激励力度大。考核目标以2022年的营收和扣非利润为基准，2023/2024/2025年相比2021年营收和扣非利润目标值为分别增长16%/32%/48%（对应同比为16%/13.8%/12.1%），触发值为分别增长12%/24%/36%（对应同比为12%/10.7%/9.7%），整体业绩目标设计稳健，进一步完善公司激励机制效果好。盈利预测与投资评级：展望后续，明年公司单店端有望迎来修复，南京工厂已投产，后续华北市场门店及团餐业务有望持续加速推进，华南市场开拓多年，单店提升和开店加速有望逐步呈现，华中市场收购项目稳步推进，单店后续有望逐步爬坡。巴比作为包子连锁头部品牌，有望受益餐饮供应链标准化提升的行业红利，团餐占比持续提升，门店端公司具备品牌和加盟连锁运营经验优势，长期开拓空间广阔。我们预计公司22-24年每股收益分别为0.83、1.04、1.33元，维持对公司的“买入”评级。风险因素：疫情反复、区域市场竞争加剧、原材料价格大幅上涨、食品安全问题。

=== Translated English ===
TITLE_EN : Barbie Food ((605338): Performance targets are solid and incentive mechanisms are improved
SUMMARY_EN: The number of restricted shares to be awarded to 

In [45]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [38]:
# Output file paths
translation_path = "equity_reports_en.csv"
logit_path       = "sentiment_logit.csv"
softmax_path     = "sentiment_softmax.csv"

processed_ids = set()

# --- Translation file setup ---
if os.path.exists(translation_path):
    df_existing = pd.read_csv(translation_path, usecols=['REPORT_ID'])
    processed_ids = set(df_existing['REPORT_ID'])
    print(f"Translation progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=["REPORT_ID", "TITLE_EN", "SUMMARY_EN"]).to_csv(translation_path, index=False)
    print("Translation file created")

# --- Logit file setup ---
if os.path.exists(logit_path):
    df_existing = pd.read_csv(logit_path, usecols=['REPORT_ID'])
    processed_ids |= set(df_existing['REPORT_ID'])
    print(f"Logit progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=[
        "REPORT_ID",
        "title_neg_logit", "title_neu_logit", "title_pos_logit",
        "summary_neg_logit", "summary_neu_logit", "summary_pos_logit"
    ]).to_csv(logit_path, index=False)
    print("Logit file created")

# --- Softmax file setup ---
if os.path.exists(softmax_path):
    df_existing = pd.read_csv(softmax_path, usecols=['REPORT_ID'])
    processed_ids |= set(df_existing['REPORT_ID'])
    print(f"Softmax progress found: {len(processed_ids)} finished")
else:
    pd.DataFrame(columns=[
        "REPORT_ID",
        "title_neg_prob", "title_neu_prob", "title_pos_prob",
        "summary_neg_prob", "summary_neu_prob", "summary_pos_prob"
    ]).to_csv(softmax_path, index=False)
    print("Softmax file created")

print(f"Total processed IDs tracked: {len(processed_ids)}")

# --- Buffers + batching setup ---
buffer_translation = []
buffer_logit = []
buffer_softmax = []

batch_size = 100
batch_bar = None

Translation progress found: 100 finished
Logit progress found: 100 finished
Softmax progress found: 100 finished
Total processed IDs tracked: 100


In [ ]:
for i, row in equity_reports.iterrows():
    ID = row['REPORT_ID']
    processed_ids = set(pd.read_csv(translation_path, usecols=["REPORT_ID"])["REPORT_ID"])

    if ID in processed_ids:
        continue

    ticker = str(row['STOCK_CODE'])
    if not (ticker.endswith('.SH') or ticker.endswith('.SZ')):
        continue

    title_cn = row['TITLE']
    summary_cn = row['SUMMARY']


    translation_dict, logit_dict, softmax_dict = process_report(
        ID, title_cn, summary_cn, pre_processing = False
    )


    buffer_translation.append(translation_dict)
    buffer_logit.append(logit_dict)
    buffer_softmax.append(softmax_dict)


    if batch_bar is None:
        batch_bar = tqdm(total=batch_size, desc="Current Batch", leave=False)

    batch_bar.update(1)

    # --- Dump batch ---
    if len(buffer_translation) >= batch_size:
        pd.DataFrame(buffer_translation).to_csv(translation_path, mode='a', header=False, index=False)
        pd.DataFrame(buffer_logit).to_csv(logit_path, mode='a', header=False, index=False)
        pd.DataFrame(buffer_softmax).to_csv(softmax_path, mode='a', header=False, index=False)

        print(f"[Row {i}] → Dumped {batch_size} rows")

        buffer_translation.clear()
        buffer_logit.clear()
        buffer_softmax.clear()

        batch_bar.close()
        batch_bar = None

# --- Final flush ---
if buffer_translation:
    pd.DataFrame(buffer_translation).to_csv(translation_path, mode='a', header=False, index=False)
    pd.DataFrame(buffer_logit).to_csv(logit_path, mode='a', header=False, index=False)
    pd.DataFrame(buffer_softmax).to_csv(softmax_path, mode='a', header=False, index=False)

    print(f"Final dump: {len(buffer_translation)} rows")

    buffer_translation.clear()
    buffer_logit.clear()
    buffer_softmax.clear()

    if batch_bar:
        batch_bar.close()

[Row 99] → Dumped 100 rows


[Row 199] → Dumped 100 rows


[Row 299] → Dumped 100 rows


[Row 399] → Dumped 100 rows


[Row 499] → Dumped 100 rows


...

[Row 3099] → Dumped 100 rows


[Row 3199] → Dumped 100 rows


[Row 3299] → Dumped 100 rows


[Row 3399] → Dumped 100 rows


Current Batch:  32%|███▏      | 32/100 [07:21<12:24, 10.95s/it]

In [ ]:
!kill -9 -1